In [1]:
# PyTorch training script for image classification using VGG16

# Mount Google Drive in Colab
from google.colab import drive
drive.mount('/content/drive')

import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torchvision.models import VGG16_Weights

# Use GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# Data directories (Google Drive paths)
train_dir = '/content/drive/MyDrive/processed_data/train'
val_dir = '/content/drive/MyDrive/processed_data/test'

# Normalization with ImageNet mean and std (for pretrained models):contentReference[oaicite:0]{index=0}
mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]

# Training data transformations: random crop, horizontal flip, color jitter, normalize
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])

# Validation data transformations: resize and center crop, normalize
val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])

# Load datasets
train_dataset = datasets.ImageFolder(train_dir, transform=train_transform)
val_dataset = datasets.ImageFolder(val_dir, transform=val_transform)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=32, shuffle=True,
                                           num_workers=4, pin_memory=torch.cuda.is_available())
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=32, shuffle=False,
                                         num_workers=4, pin_memory=torch.cuda.is_available())

# Load pretrained VGG16 (ImageNet) and fine-tune all layers:contentReference[oaicite:1]{index=1}
model = models.vgg16(weights=VGG16_Weights.DEFAULT)

# Modify classifier: add dropout before final layer (for regularization):contentReference[oaicite:2]{index=2}
num_classes = len(train_dataset.classes)
num_features = model.classifier[-1].in_features  # in_features of original classifier's last layer
classifier_layers = list(model.classifier.children())[:-1]  # all layers except final linear
classifier_layers.append(nn.Dropout(p=0.5))
classifier_layers.append(nn.Linear(num_features, num_classes))
model.classifier = nn.Sequential(*classifier_layers)

model = model.to(device)

# Loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)  # weight_decay (L2 penalty):contentReference[oaicite:3]{index=3}

# Early stopping parameters
num_epochs = 50
patience = 5
best_val_acc = 0.0
epochs_no_improve = 0

# Ensure save directory exists
os.makedirs('/content/drive/MyDrive/Model', exist_ok=True)
best_model_path = '/content/drive/MyDrive/Model/vgg16_model.pth'

# Training loop
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * inputs.size(0)
    epoch_loss = running_loss / len(train_dataset)

    # Validation
    model.eval()
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)
    val_acc = val_correct / val_total

    print(f'Epoch {epoch+1}/{num_epochs} - Loss: {epoch_loss:.4f} - Val Acc: {val_acc:.4f}')

    # Check for improvement
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        epochs_no_improve = 0
        torch.save(model.state_dict(), best_model_path)
        print(f'Validation acc improved. Saved model with acc {best_val_acc:.4f}')
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= patience:
            print(f'Early stopping after {epoch+1} epochs.')
            break

print(f'Training complete. Best Val Acc: {best_val_acc:.4f}')


Mounted at /content/drive
Using device: cuda


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:02<00:00, 195MB/s]


Epoch 1/50 - Loss: 0.4286 - Val Acc: 0.9250
Validation acc improved. Saved model with acc 0.9250
Epoch 2/50 - Loss: 0.2471 - Val Acc: 0.9525
Validation acc improved. Saved model with acc 0.9525
Epoch 3/50 - Loss: 0.2162 - Val Acc: 0.9225
Epoch 4/50 - Loss: 0.1920 - Val Acc: 0.9525
Epoch 5/50 - Loss: 0.1745 - Val Acc: 0.9525
Epoch 6/50 - Loss: 0.1892 - Val Acc: 0.9025
Epoch 7/50 - Loss: 0.1814 - Val Acc: 0.9425
Early stopping after 7 epochs.
Training complete. Best Val Acc: 0.9525
